# Gold routing

## Global threshold (child)

In [1]:
import json
from pathlib import Path

import numpy as np

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
)


# ============================================================
# Paths
# ============================================================

RESULTS_DIR = Path(
    r"C:\Users\alrazz\Downloads\Gold_routing"
)


DEV_LOGITS_FILE = (
    RESULTS_DIR
    / "dev_logits(4).npy"
)

DEV_LABELS_FILE = (
    RESULTS_DIR
    / "dev_labels(4).npy"
)


# ============================================================
# Hierarchy sizes
# Must match training
# ============================================================

NUM_PARENTS = 9


# ============================================================
# Load dev predictions
# ============================================================

print("Loading dev predictions...")


dev_logits = np.load(
    DEV_LOGITS_FILE
)


dev_labels = np.load(
    DEV_LABELS_FILE
)


print(
    "Dev logits:",
    dev_logits.shape
)

print(
    "Dev labels:",
    dev_labels.shape
)


# ============================================================
# Sigmoid
# ============================================================

def sigmoid(x):

    return (
        1.0
        /
        (
            1.0
            +
            np.exp(-x)
        )
    )


dev_probabilities = sigmoid(
    dev_logits
)


# ============================================================
# Separate parent and child labels
# ============================================================

dev_child_probabilities = (
    dev_probabilities[:, NUM_PARENTS:]
)


dev_child_labels = (
    dev_labels[:, NUM_PARENTS:]
)


print(
    "Child probability shape:",
    dev_child_probabilities.shape
)


# ============================================================
# Global threshold search
# ============================================================

thresholds = np.arange(
    0.05,
    0.96,
    0.01
)


results = []


best_threshold = None
best_f1 = -1


print("\nSearching thresholds...")


for threshold in thresholds:


    predictions = (
        dev_child_probabilities
        >= threshold
    ).astype(int)


    macro_f1 = f1_score(

        dev_child_labels,

        predictions,

        average="macro",

        zero_division=0,

    )


    micro_f1 = f1_score(

        dev_child_labels,

        predictions,

        average="micro",

        zero_division=0,

    )


    precision = precision_score(

        dev_child_labels,

        predictions,

        average="macro",

        zero_division=0,

    )


    recall = recall_score(

        dev_child_labels,

        predictions,

        average="macro",

        zero_division=0,

    )


    results.append(

        {

            "threshold": float(threshold),

            "macro_f1": float(macro_f1),

            "micro_f1": float(micro_f1),

            "precision": float(precision),

            "recall": float(recall),

        }

    )


    if macro_f1 > best_f1:

        best_f1 = macro_f1

        best_threshold = threshold



# ============================================================
# Print best result
# ============================================================

print("\n" + "=" * 80)

print("BEST GLOBAL CHILD THRESHOLD")

print("=" * 80)


print(
    "Threshold:",
    best_threshold
)


print(
    "Child Macro-F1:",
    best_f1
)






Loading dev predictions...
Dev logits: (609, 25)
Dev labels: (609, 25)
Child probability shape: (609, 16)

Searching thresholds...

BEST GLOBAL CHILD THRESHOLD
Threshold: 0.32000000000000006
Child Macro-F1: 0.7142150814195095


In [2]:
import numpy as np

from pathlib import Path

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)


# ============================================================
# Paths
# ============================================================

RESULTS_DIR = Path(
    r"C:\Users\alrazz\Downloads\Gold_routing"
)


TEST_LOGITS_FILE = (
    RESULTS_DIR
    / "test_logits.npy"
)

TEST_LABELS_FILE = (
    RESULTS_DIR
    / "test_labels.npy"
)


# ============================================================
# Fixed threshold from DEV
# ============================================================

CHILD_THRESHOLD = 0.32

NUM_PARENTS = 9


# ============================================================
# Load test predictions
# ============================================================

print("Loading test predictions...")


test_logits = np.load(
    TEST_LOGITS_FILE
)


test_labels = np.load(
    TEST_LABELS_FILE
)


print(
    "Test logits:",
    test_logits.shape
)

print(
    "Test labels:",
    test_labels.shape
)


# ============================================================
# Sigmoid
# ============================================================

def sigmoid(x):

    return (
        1.0
        /
        (
            1.0
            +
            np.exp(-x)
        )
    )


test_probabilities = sigmoid(
    test_logits
)


# ============================================================
# Separate parents and children
# ============================================================

test_parent_probabilities = (
    test_probabilities[:, :NUM_PARENTS]
)


test_child_probabilities = (
    test_probabilities[:, NUM_PARENTS:]
)


test_parent_labels = (
    test_labels[:, :NUM_PARENTS]
)


test_child_labels = (
    test_labels[:, NUM_PARENTS:]
)


# ============================================================
# Parent predictions
#
# Keep your original routing threshold
# ============================================================

PARENT_THRESHOLD = 0.5


test_parent_predictions = (
    test_parent_probabilities
    >= PARENT_THRESHOLD
).astype(int)


# ============================================================
# Child predictions
#
# IMPORTANT:
# Use tuned DEV threshold here
# ============================================================

test_child_predictions = (
    test_child_probabilities
    >= CHILD_THRESHOLD
).astype(int)


# ============================================================
# Metrics
# ============================================================

print("\n" + "="*80)
print("TEST RESULTS")
print("="*80)


# Parent

parent_macro_f1 = f1_score(
    test_parent_labels,
    test_parent_predictions,
    average="macro",
    zero_division=0,
)


parent_micro_f1 = f1_score(
    test_parent_labels,
    test_parent_predictions,
    average="micro",
    zero_division=0,
)


# Child

child_macro_f1 = f1_score(
    test_child_labels,
    test_child_predictions,
    average="macro",
    zero_division=0,
)


child_micro_f1 = f1_score(
    test_child_labels,
    test_child_predictions,
    average="micro",
    zero_division=0,
)


print(
    f"Parent Macro F1: {parent_macro_f1:.4f}"
)

print(
    f"Parent Micro F1: {parent_micro_f1:.4f}"
)


print()

print(
    f"Child Macro F1:  {child_macro_f1:.4f}"
)

print(
    f"Child Micro F1:  {child_micro_f1:.4f}"
)


# ============================================================
# Overall 25-label metrics
# ============================================================

test_predictions = np.concatenate(
    [
        test_parent_predictions,
        test_child_predictions,
    ],
    axis=1
)


overall_macro_f1 = f1_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0,
)


overall_micro_f1 = f1_score(
    test_labels,
    test_predictions,
    average="micro",
    zero_division=0,
)


print()

print(
    f"Overall Macro F1: {overall_macro_f1:.4f}"
)

print(
    f"Overall Micro F1: {overall_micro_f1:.4f}"
)


# ============================================================
# Optional classification report
# ============================================================

report = classification_report(
    test_labels,
    test_predictions,
    zero_division=0,
)


print("\nClassification report:")
print(report)


Loading test predictions...
Test logits: (633, 25)
Test labels: (633, 25)

TEST RESULTS
Parent Macro F1: 0.7804
Parent Micro F1: 0.7983

Child Macro F1:  0.7338
Child Micro F1:  0.7130

Overall Macro F1: 0.7506
Overall Micro F1: 0.7633

Classification report:
              precision    recall  f1-score   support

           0       0.91      0.65      0.75        31
           1       0.97      0.76      0.86        51
           2       0.96      0.75      0.84        61
           3       0.71      0.43      0.54        58
           4       0.83      0.80      0.82       194
           5       0.96      0.73      0.83        64
           6       0.90      0.86      0.88       226
           7       0.81      0.69      0.75       206
           8       0.82      0.71      0.76       129
           9       0.83      0.75      0.79        32
          10       0.76      0.82      0.79        79
          11       0.81      0.97      0.88        35
          12       0.75      0.80    

In [3]:
hierarchical_labels = [

    # parents
    "MT",
    "LY",
    "SP",
    "ID",
    "NA",
    "HI",
    "IN",
    "OP",
    "IP",

    # children
    "it",
    "ne",
    "sr",
    "nb",
    "re",
    "en",
    "ra",
    "dtp",
    "fi",
    "lt",
    "rv",
    "ob",
    "rs",
    "av",
    "ds",
    "ed",
]


In [4]:
report = classification_report(
    test_labels,
    test_predictions,
    target_names=hierarchical_labels,
    zero_division=0,
)

print("\nClassification report:")
print(report)



Classification report:
              precision    recall  f1-score   support

          MT       0.91      0.65      0.75        31
          LY       0.97      0.76      0.86        51
          SP       0.96      0.75      0.84        61
          ID       0.71      0.43      0.54        58
          NA       0.83      0.80      0.82       194
          HI       0.96      0.73      0.83        64
          IN       0.90      0.86      0.88       226
          OP       0.81      0.69      0.75       206
          IP       0.82      0.71      0.76       129
          it       0.83      0.75      0.79        32
          ne       0.76      0.82      0.79        79
          sr       0.81      0.97      0.88        35
          nb       0.75      0.80      0.77        30
          re       0.96      0.96      0.96        27
          en       0.73      0.83      0.78        23
          ra       0.96      0.88      0.92        26
         dtp       0.46      0.59      0.52        75
   

## Per Class Threshold tuning

In [9]:
import json
from pathlib import Path

import numpy as np

from sklearn.metrics import f1_score




# ============================================================
# Labels
# Must match your training order
# ============================================================

NUM_PARENTS = 9


child_labels = [

    "it",
    "ne",
    "sr",
    "nb",
    "re",
    "en",
    "ra",
    "dtp",
    "fi",
    "lt",
    "rv",
    "ob",
    "rs",
    "av",
    "ds",
    "ed",

]


# ============================================================
# Load data
# ============================================================

dev_logits = np.load(
    DEV_LOGITS_FILE
)


dev_labels = np.load(
    DEV_LABELS_FILE
)


# ============================================================
# Sigmoid
# ============================================================

def sigmoid(x):

    return (
        1 /
        (
            1 +
            np.exp(-x)
        )
    )


dev_probabilities = sigmoid(
    dev_logits
)


# ============================================================
# Extract children
# ============================================================

child_probabilities = (
    dev_probabilities[:, NUM_PARENTS:]
)


child_targets = (
    dev_labels[:, NUM_PARENTS:]
)


print(
    "Child shape:",
    child_probabilities.shape
)


# ============================================================
# Threshold search
# ============================================================

threshold_candidates = np.arange(
    0.05,
    0.96,
    0.01
)


best_thresholds = {}

results = {}


for i, label in enumerate(child_labels):


    print(
        "\nTuning:",
        label
    )


    best_f1 = -1
    best_threshold = 0.5


    y_true = (
        child_targets[:, i]
    )


    y_prob = (
        child_probabilities[:, i]
    )


    for threshold in threshold_candidates:


        y_pred = (
            y_prob >= threshold
        ).astype(int)


        f1 = f1_score(
            y_true,
            y_pred,
            zero_division=0,
        )


        if f1 > best_f1:

            best_f1 = f1

            best_threshold = threshold



    best_thresholds[label] = (
        float(best_threshold)
    )


    results[label] = {

        "threshold":
            float(best_threshold),

        "f1":
            float(best_f1),

        "support":
            int(y_true.sum()),

    }


    print(
        f"{label}: "
        f"threshold={best_threshold:.2f}, "
        f"F1={best_f1:.4f}, "
        f"support={int(y_true.sum())}"
    )




Child shape: (609, 16)

Tuning: it
it: threshold=0.67, F1=0.7636, support=32

Tuning: ne
ne: threshold=0.60, F1=0.8182, support=80

Tuning: sr
sr: threshold=0.77, F1=0.8788, support=35

Tuning: nb
nb: threshold=0.21, F1=0.7671, support=33

Tuning: re
re: threshold=0.05, F1=0.9200, support=26

Tuning: en
en: threshold=0.27, F1=0.7843, support=22

Tuning: ra
ra: threshold=0.31, F1=0.9231, support=26

Tuning: dtp
dtp: threshold=0.29, F1=0.4194, support=73

Tuning: fi
fi: threshold=0.28, F1=0.7407, support=30

Tuning: lt
lt: threshold=0.23, F1=0.8471, support=39

Tuning: rv
rv: threshold=0.48, F1=0.7018, support=30

Tuning: ob
ob: threshold=0.36, F1=0.5275, support=44

Tuning: rs
rs: threshold=0.22, F1=0.6742, support=39

Tuning: av
av: threshold=0.41, F1=0.3939, support=44

Tuning: ds
ds: threshold=0.95, F1=0.7957, support=52

Tuning: ed
ed: threshold=0.05, F1=0.7463, support=32


In [6]:
# ============================================================
# Save
# ============================================================

output = {

    "type":
        "per_child_thresholds",

    "thresholds":
        best_thresholds,

    "results":
        results,

}


with open(

    "best_child_thresholds_GoldRoute.json",

    "w",

    encoding="utf-8",

) as f:

    json.dump(
        output,
        f,
        indent=2,
    )


print("\nSaved thresholds:")
print(
    "best_child_thresholds_GoldRoute.json"
)


Saved thresholds:
best_child_thresholds_GoldRoute.json


In [11]:
with open(
    "best_child_thresholds_GoldRoute.json"
) as f:

    threshold_data = json.load(f)


child_thresholds = threshold_data["thresholds"]


# ============================================================
# Load test predictions
# ============================================================

test_logits = np.load(
    TEST_LOGITS_FILE
)

test_labels = np.load(
    TEST_LABELS_FILE
)


test_probabilities = sigmoid(
    test_logits
)


# ============================================================
# Split parent and child
# ============================================================

test_parent_probabilities = (
    test_probabilities[:, :NUM_PARENTS]
)


test_child_probabilities = (
    test_probabilities[:, NUM_PARENTS:]
)


# ============================================================
# Parent predictions
# Keep original routing threshold
# ============================================================

PARENT_THRESHOLD = 0.5


test_parent_predictions = (
    test_parent_probabilities
    >= PARENT_THRESHOLD
).astype(int)



# ============================================================
# Per-class child thresholds
# ============================================================

test_child_predictions = np.zeros_like(
    test_child_probabilities,
    dtype=int
)


for i, label in enumerate(child_labels):

    threshold = child_thresholds[label]

    test_child_predictions[:, i] = (
        test_child_probabilities[:, i]
        >= threshold
    ).astype(int)



# ============================================================
# Combine parent + children
# ============================================================

test_predictions = np.concatenate(
    [
        test_parent_predictions,
        test_child_predictions,
    ],
    axis=1
)



# ============================================================
# Classification report
# ============================================================

report = classification_report(

    test_labels,

    test_predictions,

    target_names=hierarchical_labels,

    zero_division=0,

)


print(report)


              precision    recall  f1-score   support

          MT       0.91      0.65      0.75        31
          LY       0.97      0.76      0.86        51
          SP       0.96      0.75      0.84        61
          ID       0.71      0.43      0.54        58
          NA       0.83      0.80      0.82       194
          HI       0.96      0.73      0.83        64
          IN       0.90      0.86      0.88       226
          OP       0.81      0.69      0.75       206
          IP       0.82      0.71      0.76       129
          it       0.83      0.75      0.79        32
          ne       0.81      0.76      0.78        79
          sr       0.88      0.80      0.84        35
          nb       0.74      0.87      0.80        30
          re       0.96      0.96      0.96        27
          en       0.70      0.83      0.76        23
          ra       0.96      0.88      0.92        26
         dtp       0.43      0.63      0.51        75
          fi       0.67    

# Hard routing

## Global threshold (child)

In [12]:
import json
from pathlib import Path

import numpy as np

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
)


# ============================================================
# Paths
# ============================================================

RESULTS_DIR = Path(
    r"C:\Users\alrazz\Downloads\Hard_routing"
)


DEV_LOGITS_FILE = (
    RESULTS_DIR
    / "dev_logits(4).npy"
)

DEV_LABELS_FILE = (
    RESULTS_DIR
    / "dev_labels(4).npy"
)


# ============================================================
# Hierarchy sizes
# Must match training
# ============================================================

NUM_PARENTS = 9


# ============================================================
# Load dev predictions
# ============================================================

print("Loading dev predictions...")


dev_logits = np.load(
    DEV_LOGITS_FILE
)


dev_labels = np.load(
    DEV_LABELS_FILE
)


print(
    "Dev logits:",
    dev_logits.shape
)

print(
    "Dev labels:",
    dev_labels.shape
)


# ============================================================
# Sigmoid
# ============================================================

def sigmoid(x):

    return (
        1.0
        /
        (
            1.0
            +
            np.exp(-x)
        )
    )


dev_probabilities = sigmoid(
    dev_logits
)


# ============================================================
# Separate parent and child labels
# ============================================================

dev_child_probabilities = (
    dev_probabilities[:, NUM_PARENTS:]
)


dev_child_labels = (
    dev_labels[:, NUM_PARENTS:]
)


print(
    "Child probability shape:",
    dev_child_probabilities.shape
)


# ============================================================
# Global threshold search
# ============================================================

thresholds = np.arange(
    0.05,
    0.96,
    0.01
)


results = []


best_threshold = None
best_f1 = -1


print("\nSearching thresholds...")


for threshold in thresholds:


    predictions = (
        dev_child_probabilities
        >= threshold
    ).astype(int)


    macro_f1 = f1_score(

        dev_child_labels,

        predictions,

        average="macro",

        zero_division=0,

    )


    micro_f1 = f1_score(

        dev_child_labels,

        predictions,

        average="micro",

        zero_division=0,

    )


    precision = precision_score(

        dev_child_labels,

        predictions,

        average="macro",

        zero_division=0,

    )


    recall = recall_score(

        dev_child_labels,

        predictions,

        average="macro",

        zero_division=0,

    )


    results.append(

        {

            "threshold": float(threshold),

            "macro_f1": float(macro_f1),

            "micro_f1": float(micro_f1),

            "precision": float(precision),

            "recall": float(recall),

        }

    )


    if macro_f1 > best_f1:

        best_f1 = macro_f1

        best_threshold = threshold



# ============================================================
# Print best result
# ============================================================

print("\n" + "=" * 80)

print("BEST GLOBAL CHILD THRESHOLD")

print("=" * 80)


print(
    "Threshold:",
    best_threshold
)


print(
    "Child Macro-F1:",
    best_f1
)






Loading dev predictions...
Dev logits: (609, 25)
Dev labels: (609, 25)
Child probability shape: (609, 16)

Searching thresholds...

BEST GLOBAL CHILD THRESHOLD
Threshold: 0.35000000000000003
Child Macro-F1: 0.6713385339291582


In [13]:
import numpy as np

from pathlib import Path

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)


# ============================================================
# Paths
# ============================================================

RESULTS_DIR = Path(
    r"C:\Users\alrazz\Downloads\Hard_routing"
)


TEST_LOGITS_FILE = (
    RESULTS_DIR
    / "test_logits.npy"
)

TEST_LABELS_FILE = (
    RESULTS_DIR
    / "test_labels.npy"
)


# ============================================================
# Fixed threshold from DEV
# ============================================================

CHILD_THRESHOLD = 0.32

NUM_PARENTS = 9


# ============================================================
# Load test predictions
# ============================================================

print("Loading test predictions...")


test_logits = np.load(
    TEST_LOGITS_FILE
)


test_labels = np.load(
    TEST_LABELS_FILE
)


print(
    "Test logits:",
    test_logits.shape
)

print(
    "Test labels:",
    test_labels.shape
)


# ============================================================
# Sigmoid
# ============================================================

def sigmoid(x):

    return (
        1.0
        /
        (
            1.0
            +
            np.exp(-x)
        )
    )


test_probabilities = sigmoid(
    test_logits
)


# ============================================================
# Separate parents and children
# ============================================================

test_parent_probabilities = (
    test_probabilities[:, :NUM_PARENTS]
)


test_child_probabilities = (
    test_probabilities[:, NUM_PARENTS:]
)


test_parent_labels = (
    test_labels[:, :NUM_PARENTS]
)


test_child_labels = (
    test_labels[:, NUM_PARENTS:]
)


# ============================================================
# Parent predictions
#
# Keep your original routing threshold
# ============================================================

PARENT_THRESHOLD = 0.5


test_parent_predictions = (
    test_parent_probabilities
    >= PARENT_THRESHOLD
).astype(int)


# ============================================================
# Child predictions
#
# IMPORTANT:
# Use tuned DEV threshold here
# ============================================================

test_child_predictions = (
    test_child_probabilities
    >= CHILD_THRESHOLD
).astype(int)


# ============================================================
# Metrics
# ============================================================

print("\n" + "="*80)
print("TEST RESULTS")
print("="*80)


# Parent

parent_macro_f1 = f1_score(
    test_parent_labels,
    test_parent_predictions,
    average="macro",
    zero_division=0,
)


parent_micro_f1 = f1_score(
    test_parent_labels,
    test_parent_predictions,
    average="micro",
    zero_division=0,
)


# Child

child_macro_f1 = f1_score(
    test_child_labels,
    test_child_predictions,
    average="macro",
    zero_division=0,
)


child_micro_f1 = f1_score(
    test_child_labels,
    test_child_predictions,
    average="micro",
    zero_division=0,
)


print(
    f"Parent Macro F1: {parent_macro_f1:.4f}"
)

print(
    f"Parent Micro F1: {parent_micro_f1:.4f}"
)


print()

print(
    f"Child Macro F1:  {child_macro_f1:.4f}"
)

print(
    f"Child Micro F1:  {child_micro_f1:.4f}"
)


# ============================================================
# Overall 25-label metrics
# ============================================================

test_predictions = np.concatenate(
    [
        test_parent_predictions,
        test_child_predictions,
    ],
    axis=1
)


overall_macro_f1 = f1_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0,
)


overall_micro_f1 = f1_score(
    test_labels,
    test_predictions,
    average="micro",
    zero_division=0,
)


print()

print(
    f"Overall Macro F1: {overall_macro_f1:.4f}"
)

print(
    f"Overall Micro F1: {overall_micro_f1:.4f}"
)


# ============================================================
# Optional classification report
# ============================================================

report = classification_report(
    test_labels,
    test_predictions,
    zero_division=0,
)


print("\nClassification report:")
print(report)


Loading test predictions...
Test logits: (633, 25)
Test labels: (633, 25)

TEST RESULTS
Parent Macro F1: 0.6966
Parent Micro F1: 0.7542

Child Macro F1:  0.7113
Child Micro F1:  0.7014

Overall Macro F1: 0.7060
Overall Micro F1: 0.7326

Classification report:
              precision    recall  f1-score   support

           0       0.94      0.55      0.69        31
           1       0.97      0.57      0.72        51
           2       0.95      0.67      0.79        61
           3       0.78      0.12      0.21        58
           4       0.85      0.79      0.82       194
           5       0.92      0.73      0.82        64
           6       0.91      0.77      0.84       226
           7       0.81      0.69      0.75       206
           8       0.81      0.53      0.64       129
           9       0.78      0.66      0.71        32
          10       0.74      0.77      0.76        79
          11       0.85      0.94      0.89        35
          12       0.84      0.87    

In [14]:
hierarchical_labels = [

    # parents
    "MT",
    "LY",
    "SP",
    "ID",
    "NA",
    "HI",
    "IN",
    "OP",
    "IP",

    # children
    "it",
    "ne",
    "sr",
    "nb",
    "re",
    "en",
    "ra",
    "dtp",
    "fi",
    "lt",
    "rv",
    "ob",
    "rs",
    "av",
    "ds",
    "ed",
]


In [15]:
report = classification_report(
    test_labels,
    test_predictions,
    target_names=hierarchical_labels,
    zero_division=0,
)

print("\nClassification report:")
print(report)



Classification report:
              precision    recall  f1-score   support

          MT       0.94      0.55      0.69        31
          LY       0.97      0.57      0.72        51
          SP       0.95      0.67      0.79        61
          ID       0.78      0.12      0.21        58
          NA       0.85      0.79      0.82       194
          HI       0.92      0.73      0.82        64
          IN       0.91      0.77      0.84       226
          OP       0.81      0.69      0.75       206
          IP       0.81      0.53      0.64       129
          it       0.78      0.66      0.71        32
          ne       0.74      0.77      0.76        79
          sr       0.85      0.94      0.89        35
          nb       0.84      0.87      0.85        30
          re       0.93      0.93      0.93        27
          en       0.76      0.96      0.85        23
          ra       0.91      0.77      0.83        26
         dtp       0.55      0.56      0.55        75
   

## Per Class Threshold tuning

In [16]:
import json
from pathlib import Path

import numpy as np

from sklearn.metrics import f1_score




# ============================================================
# Labels
# Must match your training order
# ============================================================

NUM_PARENTS = 9


child_labels = [

    "it",
    "ne",
    "sr",
    "nb",
    "re",
    "en",
    "ra",
    "dtp",
    "fi",
    "lt",
    "rv",
    "ob",
    "rs",
    "av",
    "ds",
    "ed",

]


# ============================================================
# Load data
# ============================================================

dev_logits = np.load(
    DEV_LOGITS_FILE
)


dev_labels = np.load(
    DEV_LABELS_FILE
)


# ============================================================
# Sigmoid
# ============================================================

def sigmoid(x):

    return (
        1 /
        (
            1 +
            np.exp(-x)
        )
    )


dev_probabilities = sigmoid(
    dev_logits
)


# ============================================================
# Extract children
# ============================================================

child_probabilities = (
    dev_probabilities[:, NUM_PARENTS:]
)


child_targets = (
    dev_labels[:, NUM_PARENTS:]
)


print(
    "Child shape:",
    child_probabilities.shape
)


# ============================================================
# Threshold search
# ============================================================

threshold_candidates = np.arange(
    0.05,
    0.96,
    0.01
)


best_thresholds = {}

results = {}


for i, label in enumerate(child_labels):


    print(
        "\nTuning:",
        label
    )


    best_f1 = -1
    best_threshold = 0.5


    y_true = (
        child_targets[:, i]
    )


    y_prob = (
        child_probabilities[:, i]
    )


    for threshold in threshold_candidates:


        y_pred = (
            y_prob >= threshold
        ).astype(int)


        f1 = f1_score(
            y_true,
            y_pred,
            zero_division=0,
        )


        if f1 > best_f1:

            best_f1 = f1

            best_threshold = threshold



    best_thresholds[label] = (
        float(best_threshold)
    )


    results[label] = {

        "threshold":
            float(best_threshold),

        "f1":
            float(best_f1),

        "support":
            int(y_true.sum()),

    }


    print(
        f"{label}: "
        f"threshold={best_threshold:.2f}, "
        f"F1={best_f1:.4f}, "
        f"support={int(y_true.sum())}"
    )




Child shape: (609, 16)

Tuning: it
it: threshold=0.84, F1=0.6909, support=32

Tuning: ne
ne: threshold=0.52, F1=0.7550, support=80

Tuning: sr
sr: threshold=0.36, F1=0.9041, support=35

Tuning: nb
nb: threshold=0.42, F1=0.6774, support=33

Tuning: re
re: threshold=0.95, F1=0.9412, support=26

Tuning: en
en: threshold=0.39, F1=0.7692, support=22

Tuning: ra
ra: threshold=0.21, F1=0.9615, support=26

Tuning: dtp
dtp: threshold=0.35, F1=0.4211, support=73

Tuning: fi
fi: threshold=0.27, F1=0.6087, support=30

Tuning: lt
lt: threshold=0.35, F1=0.8462, support=39

Tuning: rv
rv: threshold=0.55, F1=0.6000, support=30

Tuning: ob
ob: threshold=0.33, F1=0.4800, support=44

Tuning: rs
rs: threshold=0.33, F1=0.6400, support=39

Tuning: av
av: threshold=0.35, F1=0.3860, support=44

Tuning: ds
ds: threshold=0.82, F1=0.8454, support=52

Tuning: ed
ed: threshold=0.13, F1=0.5769, support=32


In [17]:
# ============================================================
# Save
# ============================================================

output = {

    "type":
        "per_child_thresholds",

    "thresholds":
        best_thresholds,

    "results":
        results,

}


with open(

    "best_child_thresholds_HardRoute.json",

    "w",

    encoding="utf-8",

) as f:

    json.dump(
        output,
        f,
        indent=2,
    )


print("\nSaved thresholds:")
print(
    "best_child_thresholds_HardRoute.json"
)


Saved thresholds:
best_child_thresholds_HardRoute.json


In [18]:
with open(
    "best_child_thresholds_HardRoute.json"
) as f:

    threshold_data = json.load(f)


child_thresholds = threshold_data["thresholds"]


# ============================================================
# Load test predictions
# ============================================================

test_logits = np.load(
    TEST_LOGITS_FILE
)

test_labels = np.load(
    TEST_LABELS_FILE
)


test_probabilities = sigmoid(
    test_logits
)


# ============================================================
# Split parent and child
# ============================================================

test_parent_probabilities = (
    test_probabilities[:, :NUM_PARENTS]
)


test_child_probabilities = (
    test_probabilities[:, NUM_PARENTS:]
)


# ============================================================
# Parent predictions
# Keep original routing threshold
# ============================================================

PARENT_THRESHOLD = 0.5


test_parent_predictions = (
    test_parent_probabilities
    >= PARENT_THRESHOLD
).astype(int)



# ============================================================
# Per-class child thresholds
# ============================================================

test_child_predictions = np.zeros_like(
    test_child_probabilities,
    dtype=int
)


for i, label in enumerate(child_labels):

    threshold = child_thresholds[label]

    test_child_predictions[:, i] = (
        test_child_probabilities[:, i]
        >= threshold
    ).astype(int)



# ============================================================
# Combine parent + children
# ============================================================

test_predictions = np.concatenate(
    [
        test_parent_predictions,
        test_child_predictions,
    ],
    axis=1
)



# ============================================================
# Classification report
# ============================================================

report = classification_report(

    test_labels,

    test_predictions,

    target_names=hierarchical_labels,

    zero_division=0,

)


print(report)


              precision    recall  f1-score   support

          MT       0.94      0.55      0.69        31
          LY       0.97      0.57      0.72        51
          SP       0.95      0.67      0.79        61
          ID       0.78      0.12      0.21        58
          NA       0.85      0.79      0.82       194
          HI       0.92      0.73      0.82        64
          IN       0.91      0.77      0.84       226
          OP       0.81      0.69      0.75       206
          IP       0.81      0.53      0.64       129
          it       0.91      0.62      0.74        32
          ne       0.81      0.68      0.74        79
          sr       0.87      0.94      0.90        35
          nb       0.87      0.87      0.87        30
          re       1.00      0.85      0.92        27
          en       0.76      0.96      0.85        23
          ra       0.79      0.85      0.81        26
         dtp       0.58      0.49      0.53        75
          fi       0.68    

# Soft routing

## Global threshold (child)

In [19]:
import json
from pathlib import Path

import numpy as np

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
)


# ============================================================
# Paths
# ============================================================

RESULTS_DIR = Path(
    r"C:\Users\alrazz\Downloads\Soft_routing"
)


DEV_LOGITS_FILE = (
    RESULTS_DIR
    / "dev_logits(4).npy"
)

DEV_LABELS_FILE = (
    RESULTS_DIR
    / "dev_labels(4).npy"
)


# ============================================================
# Hierarchy sizes
# Must match training
# ============================================================

NUM_PARENTS = 9


# ============================================================
# Load dev predictions
# ============================================================

print("Loading dev predictions...")


dev_logits = np.load(
    DEV_LOGITS_FILE
)


dev_labels = np.load(
    DEV_LABELS_FILE
)


print(
    "Dev logits:",
    dev_logits.shape
)

print(
    "Dev labels:",
    dev_labels.shape
)


# ============================================================
# Sigmoid
# ============================================================

def sigmoid(x):

    return (
        1.0
        /
        (
            1.0
            +
            np.exp(-x)
        )
    )


dev_probabilities = sigmoid(
    dev_logits
)


# ============================================================
# Separate parent and child labels
# ============================================================

dev_child_probabilities = (
    dev_probabilities[:, NUM_PARENTS:]
)


dev_child_labels = (
    dev_labels[:, NUM_PARENTS:]
)


print(
    "Child probability shape:",
    dev_child_probabilities.shape
)


# ============================================================
# Global threshold search
# ============================================================

thresholds = np.arange(
    0.05,
    0.96,
    0.01
)


results = []


best_threshold = None
best_f1 = -1


print("\nSearching thresholds...")


for threshold in thresholds:


    predictions = (
        dev_child_probabilities
        >= threshold
    ).astype(int)


    macro_f1 = f1_score(

        dev_child_labels,

        predictions,

        average="macro",

        zero_division=0,

    )


    micro_f1 = f1_score(

        dev_child_labels,

        predictions,

        average="micro",

        zero_division=0,

    )


    precision = precision_score(

        dev_child_labels,

        predictions,

        average="macro",

        zero_division=0,

    )


    recall = recall_score(

        dev_child_labels,

        predictions,

        average="macro",

        zero_division=0,

    )


    results.append(

        {

            "threshold": float(threshold),

            "macro_f1": float(macro_f1),

            "micro_f1": float(micro_f1),

            "precision": float(precision),

            "recall": float(recall),

        }

    )


    if macro_f1 > best_f1:

        best_f1 = macro_f1

        best_threshold = threshold



# ============================================================
# Print best result
# ============================================================

print("\n" + "=" * 80)

print("BEST GLOBAL CHILD THRESHOLD")

print("=" * 80)


print(
    "Threshold:",
    best_threshold
)


print(
    "Child Macro-F1:",
    best_f1
)






Loading dev predictions...
Dev logits: (609, 25)
Dev labels: (609, 25)
Child probability shape: (609, 16)

Searching thresholds...

BEST GLOBAL CHILD THRESHOLD
Threshold: 0.21000000000000002
Child Macro-F1: 0.6039495407454233


In [20]:
import numpy as np

from pathlib import Path

from sklearn.metrics import (
    f1_score,
    precision_score,
    recall_score,
    classification_report,
)


# ============================================================
# Paths
# ============================================================

RESULTS_DIR = Path(
    r"C:\Users\alrazz\Downloads\Soft_routing"
)


TEST_LOGITS_FILE = (
    RESULTS_DIR
    / "test_logits.npy"
)

TEST_LABELS_FILE = (
    RESULTS_DIR
    / "test_labels.npy"
)


# ============================================================
# Fixed threshold from DEV
# ============================================================

CHILD_THRESHOLD = 0.32

NUM_PARENTS = 9


# ============================================================
# Load test predictions
# ============================================================

print("Loading test predictions...")


test_logits = np.load(
    TEST_LOGITS_FILE
)


test_labels = np.load(
    TEST_LABELS_FILE
)


print(
    "Test logits:",
    test_logits.shape
)

print(
    "Test labels:",
    test_labels.shape
)


# ============================================================
# Sigmoid
# ============================================================

def sigmoid(x):

    return (
        1.0
        /
        (
            1.0
            +
            np.exp(-x)
        )
    )


test_probabilities = sigmoid(
    test_logits
)


# ============================================================
# Separate parents and children
# ============================================================

test_parent_probabilities = (
    test_probabilities[:, :NUM_PARENTS]
)


test_child_probabilities = (
    test_probabilities[:, NUM_PARENTS:]
)


test_parent_labels = (
    test_labels[:, :NUM_PARENTS]
)


test_child_labels = (
    test_labels[:, NUM_PARENTS:]
)


# ============================================================
# Parent predictions
#
# Keep your original routing threshold
# ============================================================

PARENT_THRESHOLD = 0.5


test_parent_predictions = (
    test_parent_probabilities
    >= PARENT_THRESHOLD
).astype(int)


# ============================================================
# Child predictions
#
# IMPORTANT:
# Use tuned DEV threshold here
# ============================================================

test_child_predictions = (
    test_child_probabilities
    >= CHILD_THRESHOLD
).astype(int)


# ============================================================
# Metrics
# ============================================================

print("\n" + "="*80)
print("TEST RESULTS")
print("="*80)


# Parent

parent_macro_f1 = f1_score(
    test_parent_labels,
    test_parent_predictions,
    average="macro",
    zero_division=0,
)


parent_micro_f1 = f1_score(
    test_parent_labels,
    test_parent_predictions,
    average="micro",
    zero_division=0,
)


# Child

child_macro_f1 = f1_score(
    test_child_labels,
    test_child_predictions,
    average="macro",
    zero_division=0,
)


child_micro_f1 = f1_score(
    test_child_labels,
    test_child_predictions,
    average="micro",
    zero_division=0,
)


print(
    f"Parent Macro F1: {parent_macro_f1:.4f}"
)

print(
    f"Parent Micro F1: {parent_micro_f1:.4f}"
)


print()

print(
    f"Child Macro F1:  {child_macro_f1:.4f}"
)

print(
    f"Child Micro F1:  {child_micro_f1:.4f}"
)


# ============================================================
# Overall 25-label metrics
# ============================================================

test_predictions = np.concatenate(
    [
        test_parent_predictions,
        test_child_predictions,
    ],
    axis=1
)


overall_macro_f1 = f1_score(
    test_labels,
    test_predictions,
    average="macro",
    zero_division=0,
)


overall_micro_f1 = f1_score(
    test_labels,
    test_predictions,
    average="micro",
    zero_division=0,
)


print()

print(
    f"Overall Macro F1: {overall_macro_f1:.4f}"
)

print(
    f"Overall Micro F1: {overall_micro_f1:.4f}"
)


# ============================================================
# Optional classification report
# ============================================================

report = classification_report(
    test_labels,
    test_predictions,
    zero_division=0,
)


print("\nClassification report:")
print(report)


Loading test predictions...
Test logits: (633, 25)
Test labels: (633, 25)

TEST RESULTS
Parent Macro F1: 0.7926
Parent Micro F1: 0.8047

Child Macro F1:  0.5871
Child Micro F1:  0.6052

Overall Macro F1: 0.6611
Overall Micro F1: 0.7311

Classification report:
              precision    recall  f1-score   support

           0       0.87      0.65      0.74        31
           1       0.95      0.76      0.85        51
           2       0.92      0.89      0.90        61
           3       0.63      0.53      0.58        58
           4       0.81      0.83      0.82       194
           5       0.90      0.81      0.85        64
           6       0.82      0.91      0.86       226
           7       0.76      0.77      0.76       206
           8       0.76      0.78      0.77       129
           9       0.81      0.81      0.81        32
          10       0.64      0.75      0.69        79
          11       0.81      0.86      0.83        35
          12       0.76      0.73    

In [21]:
hierarchical_labels = [

    # parents
    "MT",
    "LY",
    "SP",
    "ID",
    "NA",
    "HI",
    "IN",
    "OP",
    "IP",

    # children
    "it",
    "ne",
    "sr",
    "nb",
    "re",
    "en",
    "ra",
    "dtp",
    "fi",
    "lt",
    "rv",
    "ob",
    "rs",
    "av",
    "ds",
    "ed",
]


In [22]:
report = classification_report(
    test_labels,
    test_predictions,
    target_names=hierarchical_labels,
    zero_division=0,
)

print("\nClassification report:")
print(report)



Classification report:
              precision    recall  f1-score   support

          MT       0.87      0.65      0.74        31
          LY       0.95      0.76      0.85        51
          SP       0.92      0.89      0.90        61
          ID       0.63      0.53      0.58        58
          NA       0.81      0.83      0.82       194
          HI       0.90      0.81      0.85        64
          IN       0.82      0.91      0.86       226
          OP       0.76      0.77      0.76       206
          IP       0.76      0.78      0.77       129
          it       0.81      0.81      0.81        32
          ne       0.64      0.75      0.69        79
          sr       0.81      0.86      0.83        35
          nb       0.76      0.73      0.75        30
          re       0.96      0.85      0.90        27
          en       1.00      0.04      0.08        23
          ra       0.92      0.88      0.90        26
         dtp       0.43      0.36      0.39        75
   

## Per Class Threshold tuning

In [23]:
import json
from pathlib import Path

import numpy as np

from sklearn.metrics import f1_score




# ============================================================
# Labels
# Must match your training order
# ============================================================

NUM_PARENTS = 9


child_labels = [

    "it",
    "ne",
    "sr",
    "nb",
    "re",
    "en",
    "ra",
    "dtp",
    "fi",
    "lt",
    "rv",
    "ob",
    "rs",
    "av",
    "ds",
    "ed",

]


# ============================================================
# Load data
# ============================================================

dev_logits = np.load(
    DEV_LOGITS_FILE
)


dev_labels = np.load(
    DEV_LABELS_FILE
)


# ============================================================
# Sigmoid
# ============================================================

def sigmoid(x):

    return (
        1 /
        (
            1 +
            np.exp(-x)
        )
    )


dev_probabilities = sigmoid(
    dev_logits
)


# ============================================================
# Extract children
# ============================================================

child_probabilities = (
    dev_probabilities[:, NUM_PARENTS:]
)


child_targets = (
    dev_labels[:, NUM_PARENTS:]
)


print(
    "Child shape:",
    child_probabilities.shape
)


# ============================================================
# Threshold search
# ============================================================

threshold_candidates = np.arange(
    0.05,
    0.96,
    0.01
)


best_thresholds = {}

results = {}


for i, label in enumerate(child_labels):


    print(
        "\nTuning:",
        label
    )


    best_f1 = -1
    best_threshold = 0.5


    y_true = (
        child_targets[:, i]
    )


    y_prob = (
        child_probabilities[:, i]
    )


    for threshold in threshold_candidates:


        y_pred = (
            y_prob >= threshold
        ).astype(int)


        f1 = f1_score(
            y_true,
            y_pred,
            zero_division=0,
        )


        if f1 > best_f1:

            best_f1 = f1

            best_threshold = threshold



    best_thresholds[label] = (
        float(best_threshold)
    )


    results[label] = {

        "threshold":
            float(best_threshold),

        "f1":
            float(best_f1),

        "support":
            int(y_true.sum()),

    }


    print(
        f"{label}: "
        f"threshold={best_threshold:.2f}, "
        f"F1={best_f1:.4f}, "
        f"support={int(y_true.sum())}"
    )




Child shape: (609, 16)

Tuning: it
it: threshold=0.23, F1=0.8000, support=32

Tuning: ne
ne: threshold=0.09, F1=0.6502, support=80

Tuning: sr
sr: threshold=0.40, F1=0.9014, support=35

Tuning: nb
nb: threshold=0.25, F1=0.7097, support=33

Tuning: re
re: threshold=0.11, F1=0.9388, support=26

Tuning: en
en: threshold=0.18, F1=0.5455, support=22

Tuning: ra
ra: threshold=0.22, F1=0.8679, support=26

Tuning: dtp
dtp: threshold=0.05, F1=0.3616, support=73

Tuning: fi
fi: threshold=0.17, F1=0.7097, support=30

Tuning: lt
lt: threshold=0.43, F1=0.7941, support=39

Tuning: rv
rv: threshold=0.21, F1=0.5965, support=30

Tuning: ob
ob: threshold=0.28, F1=0.4471, support=44

Tuning: rs
rs: threshold=0.24, F1=0.5116, support=39

Tuning: av
av: threshold=0.07, F1=0.3066, support=44

Tuning: ds
ds: threshold=0.41, F1=0.7619, support=52

Tuning: ed
ed: threshold=0.11, F1=0.6557, support=32


In [24]:
# ============================================================
# Save
# ============================================================

output = {

    "type":
        "per_child_thresholds",

    "thresholds":
        best_thresholds,

    "results":
        results,

}


with open(

    "best_child_thresholds_SoftRoute.json",

    "w",

    encoding="utf-8",

) as f:

    json.dump(
        output,
        f,
        indent=2,
    )


print("\nSaved thresholds:")
print(
    "best_child_thresholds_SoftRoute.json"
)


Saved thresholds:
best_child_thresholds_SoftRoute.json


In [25]:
with open(
    "best_child_thresholds_SoftRoute.json"
) as f:

    threshold_data = json.load(f)


child_thresholds = threshold_data["thresholds"]


# ============================================================
# Load test predictions
# ============================================================

test_logits = np.load(
    TEST_LOGITS_FILE
)

test_labels = np.load(
    TEST_LABELS_FILE
)


test_probabilities = sigmoid(
    test_logits
)


# ============================================================
# Split parent and child
# ============================================================

test_parent_probabilities = (
    test_probabilities[:, :NUM_PARENTS]
)


test_child_probabilities = (
    test_probabilities[:, NUM_PARENTS:]
)


# ============================================================
# Parent predictions
# Keep original routing threshold
# ============================================================

PARENT_THRESHOLD = 0.5


test_parent_predictions = (
    test_parent_probabilities
    >= PARENT_THRESHOLD
).astype(int)



# ============================================================
# Per-class child thresholds
# ============================================================

test_child_predictions = np.zeros_like(
    test_child_probabilities,
    dtype=int
)


for i, label in enumerate(child_labels):

    threshold = child_thresholds[label]

    test_child_predictions[:, i] = (
        test_child_probabilities[:, i]
        >= threshold
    ).astype(int)



# ============================================================
# Combine parent + children
# ============================================================

test_predictions = np.concatenate(
    [
        test_parent_predictions,
        test_child_predictions,
    ],
    axis=1
)



# ============================================================
# Classification report
# ============================================================

report = classification_report(

    test_labels,

    test_predictions,

    target_names=hierarchical_labels,

    zero_division=0,

)


print(report)


              precision    recall  f1-score   support

          MT       0.87      0.65      0.74        31
          LY       0.95      0.76      0.85        51
          SP       0.92      0.89      0.90        61
          ID       0.63      0.53      0.58        58
          NA       0.81      0.83      0.82       194
          HI       0.90      0.81      0.85        64
          IN       0.82      0.91      0.86       226
          OP       0.76      0.77      0.76       206
          IP       0.76      0.78      0.77       129
          it       0.74      0.81      0.78        32
          ne       0.54      0.81      0.65        79
          sr       0.86      0.86      0.86        35
          nb       0.73      0.73      0.73        30
          re       0.85      0.85      0.85        27
          en       0.47      0.61      0.53        23
          ra       0.74      0.88      0.81        26
         dtp       0.29      0.76      0.42        75
          fi       0.65    